In [ ]:
# Code écrit par Claude (Anthropic, Claude Opus 5.5) pour la compatibilité Google Colab.
# Adapté de la cellule d'installation de demo_2/notebooks/fr/07_llm_and_rag_FP16.ipynb (IFT 6758).
# OPTIONNEL : une fois par nouveau runtime Colab (version 2025.07 = Python 3.11, GPU T4).
INSTALL_COLAB = False  # Mettre à True pour installer, puis remettre à False

import sys
import subprocess
from pathlib import Path
from importlib.metadata import version

REPO_URL = "https://github.com/Mahdi267/NHL-data-analysis.git"
REPO_DIR = Path("/content/NHL-data-analysis")
REQ_FILE = Path("/content/requirements_colab.txt")

# Détecte Colab ; sinon on utilise le .venv local
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print("Environnement local : installation sautée (uv sync --extra llm dans le terminal).")
elif not INSTALL_COLAB:
    print("Installation sautée. Mettre INSTALL_COLAB = True sur un nouveau runtime Colab.")
else:
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError("Python 3.11 requis : Exécution → Modifier le type d'exécution → version 2025.07.")

    # 1. Code du projet (dépôt public : aucun identifiant)
    if not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
    else:
        subprocess.check_call(["git", "-C", str(REPO_DIR), "pull"])

    # 2. Mêmes versions qu'en local (uv.lock), sauf torch : on garde celui de Colab,
    #    compilé pour son pilote CUDA (le torch du lock est une version CUDA 13)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    export = subprocess.check_output(
        [sys.executable, "-m", "uv", "export", "--frozen", "--extra", "llm",
         "--no-dev", "--no-hashes", "--no-emit-project"],
        cwd=REPO_DIR, text=True,
    )
    skip = ("torch==", "triton==", "nvidia-")
    lines = [line for line in export.splitlines() if not line.startswith(skip)]
    lines.append(f"torch=={version('torch').split('+')[0]}")  # Fige le torch déjà installé
    REQ_FILE.write_text("\n".join(lines))

    # 3. Installation dans le Python du notebook, puis le projet en mode éditable
    uv_install = [sys.executable, "-m", "uv", "pip", "install", "--python", sys.executable]
    subprocess.check_call(uv_install + ["-r", str(REQ_FILE)])
    subprocess.check_call(uv_install + ["--no-deps", "-e", str(REPO_DIR)])
    print("Installation terminée. Exécution → Redémarrer la session, puis remettre INSTALL_COLAB = False.")

In [ ]:
# Code écrit par Claude (Anthropic, Claude Opus 5.5) pour la compatibilité Google Colab.
import os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Adapter au dossier de CE notebook (notebooks/ ou visualizations/) : les chemins relatifs en dépendent
NOTEBOOK_DIR = Path("/content/NHL-data-analysis/ift6758/notebooks")
if IN_COLAB:
    os.chdir(NOTEBOOK_DIR)
print("Dossier de travail :", Path.cwd())

from ift6758.llm.qwen_utils import check_env
check_env()  # Doit afficher Python 3.11 et CUDA GPU: Tesla T4

In [1]:
from ift6758.features.tidy import json2tidy
from pathlib import Path
import pandasai as pai
from pandasai.llm.fake import FakeLLM

game_path = Path().resolve().parent / "data" / "2022" / "2022030411.json"
print(game_path, game_path.exists())

dataframe = json2tidy(str(game_path), True)
print(dataframe.shape)
print(dataframe.columns.tolist())
print(dataframe)


/Users/jcforget/Maitrise/Cours/IFT 6758/Projet/Github/NHL-data-analysis/ift6758/data/raw/2022030411.json True
(69, 27)
['game_id', 'event_id', 'season', 'situation_code', 'period', 'time', 'shooter_id', 'scorer_id', 'event_type', 'x_coord', 'y_coord', 'goalie_id', 'shot_type', 'team_id', 'homeTeam_id', 'homeTeamDefendingSide', 'game_type', 'shooter', 'shooter_last_name', 'scorer', 'scorer_last_name', 'goalie', 'goalie_last_name', 'emptyNet', 'strength', 'away_players', 'home_players']
       game_id  event_id    season situation_code  period   time  shooter_id  \
0   2022030411        55  20222023           1551       1  01:29   8479372.0   
1   2022030411        62  20222023           1551       1  03:32   8478055.0   
2   2022030411        64  20222023           1551       1  03:49   8479353.0   
3   2022030411        66  20222023           1551       1  03:52   8478020.0   
4   2022030411        67  20222023           1551       1  03:57   8474166.0   
..         ...       ...      

In [2]:
# On enveloppe le dataframe en tableDuck
# https://docs.pandas-ai.com/v3/migration-guide#step-4-migrate-dataframes-optional
df_pai = pai.DataFrame(dataframe)
table = df_pai.schema.name
print("SQL table name :", table)

# https://pandasai-docs.readthedocs.io/en/latest/API/llms/#pandasai.llm.fake.FakeLLM
# Create à samll code for pandas-ai to run. If code bad, returns results and error to llm up to 3 times
code_fake = f'''
df = execute_sql_query("SELECT COUNT(*) AS n FROM {table} WHERE event_type = 'goal'")
result = {{"type": "number", "value": int(df["n"][0])}}
'''

# False LLm configuration
llm = FakeLLM(output=code_fake)
pai.config.set({"llm": llm})

reponse = df_pai.chat("How many goals are in this game?") # Our question
print("Réponse :", reponse, type(reponse))

# Answer that Qwen will receive
print(llm.last_prompt)


SQL table name : table_2df8dc309941b722d5dddbc4623a992f
Réponse : 7 <class 'pandasai.core.response.number.NumberResponse'>
<tables>

<table dialect="duckdb" table_name="table_2df8dc309941b722d5dddbc4623a992f" columns="[{"name": "game_id", "type": "string", "description": null, "expression": null, "alias": null}, {"name": "event_id", "type": "integer", "description": null, "expression": null, "alias": null}, {"name": "season", "type": "string", "description": null, "expression": null, "alias": null}, {"name": "situation_code", "type": "string", "description": null, "expression": null, "alias": null}, {"name": "period", "type": "integer", "description": null, "expression": null, "alias": null}, {"name": "time", "type": "string", "description": null, "expression": null, "alias": null}, {"name": "shooter_id", "type": "float", "description": null, "expression": null, "alias": null}, {"name": "scorer_id", "type": "float", "description": null, "expression": null, "alias": null}, {"name": "eve

Pandais-ai construit le prompt avec un gabarit. Il prend des gabarits fixes et les remplit avec le nom de la table, la liste des colonnes, head() et ta question. C’est du simple remplissage de texte, comme une f-string.

In [3]:
from pandasai.llm.base import LLM
from ift6758.llm.qwen_utils import load_qwen, build_messages, generate

# Adaptation of pandasai/llm/fake.py
class QwenLLM(LLM):
    def __init__(self, model, tokenizer, max_new_tokens=1024):
        # Initialise call counter for every new question
        self.model = model
        self.tokenizer = tokenizer
        self.max_new_tokens = max_new_tokens
        self.n_calls = 0
        self.last_prompt = None
        self.type

    def call(self, instruction, context=None) -> str:
        self.n_calls +=1
        self.last_prompt = instruction.to_string()
        answer = generate(model=self.model,tokenizer=self.tokenizer,messages=build_messages(self.last_prompt,context=None,system=None),max_new_tokens=self.max_new_tokens,do_sample=False)
        return answer["answer"]

    @property
    def type(self) -> str:
        return "qwen_local"

In [4]:
model, tokenizer, device = load_qwen()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [6]:
df = pai.DataFrame(dataframe)
table_test = df.schema.name
print("SQL table name :", table_test)
qwen = QwenLLM(model,tokenizer)
pai.config.set({"llm": qwen})
reponse = df.chat("How many goals are in this game?") # Our question
print("Réponse :", reponse, type(reponse))


SQL table name : table_2df8dc309941b722d5dddbc4623a992f
Réponse : 62 <class 'pandasai.core.response.number.NumberResponse'>


Il y a 62 shot-on-goal et 7 goal. Si le modèle avait afficher 69, j'aurais pu comprendre, mais la je ne comprend pas, il compte juste les shot-on-goal et non pas tout els event avec goal.

In [7]:
print(qwen.n_calls)
print(df._agent.last_generated_code)

1
import pandas as pd
sql_query = """
SELECT COUNT(*) AS goal_count
FROM table_2df8dc309941b722d5dddbc4623a992f
WHERE event_type = 'shot-on-goal'
"""
goal_data = execute_sql_query(sql_query)
goal_count = goal_data.iloc[0]['goal_count']
result = {'type': 'number', 'value': goal_count}


Okay, on voit bien ici qu'il a écrit dans son code where event_typ == 'shot-on-goal'. Puisque le code roule, panda ne la pas corrigé. Pandas corrige juste les mauvais code. Donc, on peut donner plus d'infos à pandas et a qwen en ajoutant une description à mon event_id. Si ca marche, on va l'ajouter à toutes les autres colonnes du dataframe.

In [13]:
for c in (1,len(df.schema.columns)-1):
    if df.schema.columns[c].name == "event_id":
        print(df.schema.columns[c])
        df.schema.columns[c].description = "Type of event: 'goal' (a scored goal) or 'shot-on-goal' (a shot stopped by the goalie, NOT a goal)."
        print(df.schema.columns[c])

name='event_id' type='integer' description="Type of event: 'goal' (a scored goal) or 'shot-on-goal' (a shot stopped by the goalie, NOT a goal)." expression=None alias=None
name='event_id' type='integer' description="Type of event: 'goal' (a scored goal) or 'shot-on-goal' (a shot stopped by the goalie, NOT a goal)." expression=None alias=None


On réesseaie le prompt

In [14]:
reponse = df.chat("How many goals are in this game?") # Our question
print("Réponse :", reponse, type(reponse))

Réponse : 7 <class 'pandasai.core.response.number.NumberResponse'>


Bingo, on applique donc une description pour chaque colonne, c'est

In [15]:
# Table de description faites avec
# Sources : ift6758/features/tidy.py (Mahdi), ift6758/features/Q1_P3.txt, NHLyear.download_all (William),
# profil de la saison 2021-22 (89 585 lignes) et types DuckDB vérifiés le 2026-10-17.
dic_des = {
    "game_id": "Integer game identifier, 10 digits: 4-digit season start year + 2-digit game type (02 = regular season, 03 = playoffs) + 4-digit game number. Regular season: game number 0001 to 1312, e.g. game 50 of the 2021-22 regular season is 2021020050. Playoffs: the last 4 digits are 0 + round + matchup + game, e.g. 2018030212 = round 4, matchup 1, game 1; game 212 of the 2018-19 playoffs is 2018030212.",
    "event_id": "Integer identifier of the event inside its game. Unique only within one game_id.",
    "season": "Integer season identifier: start year followed by end year, e.g. 20212022 for the 2021-22 season.",
    "situation_code": "4-character text code of the players on ice: away goalie (1 in net, 0 pulled), away skaters, home skaters, home goalie. Example: '1451' = away 4 skaters vs home 5 skaters, both goalies in net.",
    "period": "Integer period number: 1 to 3 are regulation, 4 is overtime. In the regular season, period 5 is the shootout; in the playoffs, periods 5 and more are extra overtimes.",
    "time": "Text 'MM:SS', time elapsed in the period when the event happened.",
    "shooter_id": "Player id of the player who took the shot. For goals it is the scorer, so shooter_id covers every row and is the column to use for a player's shots and goal percentage.",
    "scorer_id": "Player id of the goal scorer. Only filled when event_type is 'goal', empty for shots.",
    "event_type": "Type of event: 'goal' (a scored goal) or 'shot-on-goal' (a shot stopped by the goalie, NOT a goal).",
    "x_coord": "X coordinate of the shot in feet, from -99 to 99. Center ice is 0; the nets are near x = -89 and x = 89.",
    "y_coord": "Y coordinate of the shot in feet, from -42 to 42. 0 is the middle of the ice.",
    "goalie_id": "Player id of the goalie in net who faced the shot. Empty when the net was empty.",
    "shot_type": "Type of shot: 'wrist', 'snap', 'slap', 'backhand', 'tip-in', 'deflected', 'wrap-around'.",
    "team_id": "Integer id of the team that took the shot. Team ids: 1 NJD New Jersey, 2 NYI NY Islanders, 3 NYR NY Rangers, 4 PHI Philadelphia, 5 PIT Pittsburgh, 6 BOS Boston, 7 BUF Buffalo, 8 MTL Montreal, 9 OTT Ottawa, 10 TOR Toronto, 12 CAR Carolina, 13 FLA Florida, 14 TBL Tampa Bay, 15 WSH Washington, 16 CHI Chicago, 17 DET Detroit, 18 NSH Nashville, 19 STL St. Louis, 20 CGY Calgary, 21 COL Colorado, 22 EDM Edmonton, 23 VAN Vancouver, 24 ANA Anaheim, 25 DAL Dallas, 26 LAK Los Angeles, 28 SJS San Jose, 29 CBJ Columbus, 30 MIN Minnesota, 52 WPG Winnipeg, 53 ARI Arizona, 54 VGK Vegas, 55 SEA Seattle.",
    "homeTeam_id": "Integer id of the home team of the game. A row is a home team shot when team_id = homeTeam_id.",
    "homeTeamDefendingSide": "Side of the rink defended by the home team during this period: 'left' or 'right'.",
    "game_type": "Type of game: 'regular season' or 'playoffs' (a game of a playoff series).",
    "shooter": "First name only of the shooter. The last name is in shooter_last_name.",
    "shooter_last_name": "Last name of the shooter.",
    "scorer": "First name only of the goal scorer, empty for shots. The last name is in scorer_last_name.",
    "scorer_last_name": "Last name of the goal scorer, empty for shots.",
    "goalie": "First name only of the goalie in net, empty when the net was empty. The last name is in goalie_last_name.",
    "goalie_last_name": "Last name of the goalie in net, empty when the net was empty.",
    "emptyNet": "Boolean: True when no goalie was in net (goalie pulled), False otherwise.",
    "strength": "Strength of the shooting team, ONLY for goals: 'even', 'power play' or 'shorthanded'. Empty for every shot, SO IT CANNOT be used to count shots or to compute a goal percentage by strength.",
    "away_players": "Integer number of away players on ice (goalie + skaters), from situation_code. A value of 1 means a shootout attempt or a penalty shot.",
    "home_players": "Integer number of home players on ice (goalie + skaters), from situation_code. A value of 1 means a shootout attempt or a penalty shot.",
}
for col in df.schema.columns:
    if col.name in dic_des:
        col.description = dic_des[col.name]
        print(col)

name='game_id' type='string' description='Integer game identifier, 10 digits: 4-digit season start year + 2-digit game type (02 = regular season, 03 = playoffs) + 4-digit game number. Regular season: game number 0001 to 1312, e.g. game 139 of the 2023-24 regular season is 2023020139. Playoffs: the last 4 digits are 0 + round + matchup + game, e.g. 2022030411 = round 4, matchup 1, game 1; game 411 of the 2019-20 playoffs is 2019030411.' expression=None alias=None
name='event_id' type='integer' description='Integer identifier of the event inside its game. Unique only within one game_id.' expression=None alias=None
name='season' type='string' description='Integer season identifier: start year followed by end year, e.g. 20212022 for the 2021-22 season.' expression=None alias=None
name='situation_code' type='string' description="4-character text code of the players on ice: away goalie (1 in net, 0 pulled), away skaters, home skaters, home goalie. Example: '1451' = away 4 skaters vs home 5 s

Alright, il faut maintenant applquer les régles de l'énoncer en système prompt

In [ ]:
# Instructions tiré de l'énoncé
table_description = (
    "NHL play-by-play data: one row per shot on goal or goal (missed shots, blocked shots, passes, assists, "
    "penalties and hits are not recorded). Goal percentage = number of 'goal' rows / total number of rows "
    "(goals + shots on goal) for the selected group. Teams are identified only by team_id (see its description). "
    "If a question needs information that is not in this table, answer with type 'string' and the value: "
    "'I cannot answer this question with the available data.'"
)

In [ ]:
import time
import pandas as pd
from pathlib import Path
from IPython.display import Image, display
from ift6758.features.simple_visualizations import games_list

# Code écrit par Claude (Anthropic, Claude Opus 5.5)
# Contournement d'un bug de PandasAI 3.0.0 : il enregistre les graphiques dans
# <dossier courant>/exports/charts mais ne crée ce dossier qu'à la racine du projet
Path("exports/charts").mkdir(parents=True, exist_ok=True)

SEASONS = [2016, 2018, 2020, 2022]  # Seasons used by training questions
data = pd.concat([games_list(s) for s in SEASONS], ignore_index=True)  # ~30 s
print(data.shape)
print(data.groupby(["season", "game_type"])["game_id"].nunique())  # plz check if seasons are complete. Lots of mistakes sometimes

In [ ]:
def make_pai_df(data):
    """Creates a new pandasai dataframe with columns description

    New object for each question : no data from previous questions.
    """
    df_q = pai.DataFrame(data)
    for col in df_q.schema.columns:
        if col.name in dic_des:
            col.description = dic_des[col.name]
    df_q.schema.description = table_description
    return df_q